## MLflow Churn Model — Binary Classification
Training two models (Logistic Regression vs Random Forest) to predict 
customer churn using RFM features from the Silver layer.

Experiments tracked via MLflow:
- Parameters logged: model type, hyperparameters
- Metrics logged: accuracy, AUC, precision, recall
- Best model promoted to MLflow Model Registry

In [0]:
# MLFLOW CHURN MODEL — Binary Classification

SILVER_PATH = "/Volumes/workspace/default/raw_data/silver"

In [0]:
# Load Silver RFM feature table
customer_rfm = spark.read.format("delta") \
    .load(f"{SILVER_PATH}/customer_rfm_features")

print("Silver RFM table loaded")
print(f"Total customers: {customer_rfm.count()}")
print(f"Columns: {customer_rfm.columns}")

In [0]:
# Checking class balance before training
total = customer_rfm.count()
churned = customer_rfm.filter(customer_rfm.is_churned == 1).count()
active = customer_rfm.filter(customer_rfm.is_churned == 0).count()

print("CLASS DISTRIBUTION")
print(f"Total customers : {total}")
print(f"Churned (1)     : {churned} ({round(churned/total*100, 1)}%)")
print(f"Active  (0)     : {active} ({round(active/total*100, 1)}%)")
print("")
print("Class balance acceptable for binary classification")

In [0]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# Selecting features
feature_cols = [
    "frequency", 
    "monetary_value",
    "avg_items_per_order",
    "avg_freight_value",
    "customer_state"
]

# Convert to Pandas
df = customer_rfm.select(feature_cols + ["is_churned"]) \
    .dropna().toPandas()

print("DATASET SHAPE")
print(f"Rows    : {df.shape[0]}")
print(f"Columns : {df.shape[1]}")

# Encode categorical column
le = LabelEncoder()
df["customer_state_encoded"] = le.fit_transform(df["customer_state"])
df = df.drop(columns=["customer_state"])

print("")
print("")
print("FEATURE SUMMARY AFTER ENCODING")
print(df.describe())


In [0]:
# Check feature correlation with target before training
print("FEATURE CORRELATION WITH is_churned")
print("")
correlations = df.corr()["is_churned"].drop("is_churned") \
    .sort_values(ascending=False)
print(correlations.to_string())
print("")

In [0]:
# Data spliting
X = df.drop(columns=["is_churned"])
y = df["is_churned"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [0]:
print("TRAIN / TEST SPLIT")
print(f"Training rows : {X_train.shape[0]}")
print(f"Test rows     : {X_test.shape[0]}")
print(f"Features      : {X_train.shape[1]}")
print("")

print("CHURN RATE IN EACH SPLIT :- ")
print(f"Train churn rate : {round(y_train.mean()*100, 1)}%")
print(f"Test churn rate  : {round(y_test.mean()*100, 1)}%")

Train Model 1 — Logistic Regression with MLflow

In [0]:
# Get your Databricks username
username = spark.sql("SELECT current_user()").collect()[0][0]


In [0]:
import mlflow
import mlflow.sklearn
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, roc_auc_score, 
                             precision_score, recall_score)

username = spark.sql("SELECT current_user()").collect()[0][0]
mlflow.set_experiment(f"/Users/{username}/churn_prediction_olist")

with mlflow.start_run(run_name="logistic_regression_v1"):
    
    # Train
    lr = LogisticRegression(max_iter=1000, random_state=42)
    lr.fit(X_train, y_train)
    y_pred_lr = lr.predict(X_test)
    
    # Metrics
    acc  = accuracy_score(y_test, y_pred_lr)
    auc  = roc_auc_score(y_test, lr.predict_proba(X_test)[:,1])
    prec = precision_score(y_test, y_pred_lr)
    rec  = recall_score(y_test, y_pred_lr)
    
    # Log to MLflow
    mlflow.log_param("model_type", "logistic_regression")
    mlflow.log_param("max_iter", 1000)
    mlflow.log_metric("accuracy", acc)
    mlflow.log_metric("auc", auc)
    mlflow.log_metric("precision", prec)
    mlflow.log_metric("recall", rec)
    mlflow.sklearn.log_model(lr, "logistic_regression_model")
    
    print("LOGISTIC REGRESSION RESULTS :-")
    print(f"Accuracy  : {round(acc, 4)}")
    print(f"AUC       : {round(auc, 4)}")
    print(f"Precision : {round(prec, 4)}")
    print(f"Recall    : {round(rec, 4)}")